# Génération des champs manquants - OpenRouter

Ce notebook génère les 3 champs manquants du dataset :
- `description` (obligatoire)
- `prix` (obligatoire)
- `matiere` (recommandé)

## 1. Installation des dépendances

In [12]:
!pip install openai requests tqdm -q

import pandas as pd
import requests
import time
from tqdm import tqdm
import json
from pathlib import Path
import random
from openai import OpenAI

print("✅ Dépendances importées")

✅ Dépendances importées


## 2. Configuration des chemins

In [13]:
# Chemins des fichiers
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
INPUT_FILE = "products_partial.csv"
OUTPUT_FILE = "products_final.csv"
PROGRESS_FILE = "products_progress.csv"

print(f"📁 Répertoire de travail : {BASE_DIR}")
print(f"   Fichier d'entrée : {INPUT_FILE}")
print(f"   Fichier de sortie : {OUTPUT_FILE}")

📁 Répertoire de travail : ..
   Fichier d'entrée : products_partial.csv
   Fichier de sortie : products_final.csv


## 3. Chargement du CSV partiel

In [14]:
# Charger le fichier
input_path = DATA_DIR / INPUT_FILE
products_df = pd.read_csv(input_path)

print(f"✅ CSV chargé : {len(products_df)} produits")
print(f"   Colonnes : {list(products_df.columns)}")
print(f"\n📊 Aperçu:")
products_df.head(3)

✅ CSV chargé : 400 produits
   Colonnes : ['id', 'nom', 'categorie', 'sous_categorie', 'couleur', 'image_path']

📊 Aperçu:


,id,nom,categorie,sous_categorie,couleur,image_path
0,15832,Visudh Pink Printed Ethnic Kurta,Topwear,Kurtas,Pink,images/15832.jpg
1,33968,Femella Women Gold Shirt,Topwear,Shirts,Gold,images/33968.jpg
2,3314,Guerrilla Men's Warrior Brown T-shirt,Topwear,Tshirts,Brown,images/3314.jpg


## 4. Configuration API z.ai

In [15]:
# Configuration API z.ai (compatible OpenAI)
ZAI_API_KEY = input("Entrez votre clé API z.ai: ")

# Initialiser le client OpenAI avec l'endpoint z.ai
client = OpenAI(
    api_key=ZAI_API_KEY,
    base_url="https://api.z.ai/api/coding/paas/v4"
)

# Choix du modèle
print("\nChoisissez le modèle :")
print("1. glm-4.7 (standard, tâches complexes)")
print("2. glm-4.5-air (léger, réponses plus rapides)")

model_choice = input("Votre choix (1 ou 2): ")
MODEL_NAME = "glm-4.7" if model_choice == "1" else "glm-4.5-air"

print(f"\n✅ Configuration terminée")
print(f"   Modèle : {MODEL_NAME}")
print(f"   Endpoint : https://api.z.ai/api/coding/paas/v4")


Choisissez le modèle :
1. glm-4.7 (standard, tâches complexes)
2. glm-4.5-air (léger, réponses plus rapides)

✅ Configuration terminée
   Modèle : glm-4.5-air
   Endpoint : https://api.z.ai/api/coding/paas/v4


## 5. Système de génération par batch avec gestion d'erreur

In [16]:
def call_zai(prompt, max_retries=3):
    """Appel API z.ai via client OpenAI avec retry"""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=150,
                temperature=0.7,
                extra_body={  # ✅ DÉSACTIVER LE MODE THINKING
                    "thinking": {
                        "type": "disabled"
                    }
                }
            )
            
            # Extraire le contenu de la réponse
            if response.choices and len(response.choices) > 0:
                content = response.choices[0].message.content
                if content and content.strip():
                    return content.strip()
                else:
                    print(f"   ⚠️ Réponse vide de l'API")
                    return None
            else:
                print(f"   ⚠️ Pas de choices dans la réponse")
                return None
                
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"   ⚠️ Erreur (tentative {attempt+1}/{max_retries}): {str(e)[:80]}")
                # Backoff exponentiel
                time.sleep(2 ** attempt)
            else:
                print(f"   ❌ Échec après {max_retries} tentatives: {str(e)[:80]}")
                return None

                
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"   ⚠️ Erreur (tentative {attempt+1}/{max_retries}): {str(e)[:80]}")
                # Backoff exponentiel
                time.sleep(2 ** attempt)
            else:
                print(f"   ❌ Échec après {max_retries} tentatives: {str(e)[:80]}")
                return None

def generate_description(row):
    """Génère une description de produit"""
    prompt = f"""Génère une description de 2-3 phrases pour ce produit de mode:

Nom: {row['nom']}
Catégorie: {row['categorie']}
Sous-catégorie: {row['sous_categorie']}
Couleur: {row['couleur']}

La description doit être:
- En français
- Claire et concise
- Mettre en valeur les caractéristiques principales
- Adaptée à un site e-commerce
- 2-3 phrases maximum

Description:"""
    return call_zai(prompt)

def generate_price(row):
    """Génère un prix réaliste en FCFA"""
    # Prix de base par catégorie
    base_prices = {
        'Topwear': 15000,
        'Shoes': 35000,
        'Watches': 45000,
        'Bags': 28000,
        'Bottomwear': 18000,
        'Jewellery': 12000,
        'Eyewear': 15000,
        'Innerwear': 8000
    }
    
    base = base_prices.get(row['categorie'], 20000)
    # Variation de ±30%
    price = int(base * (0.7 + 0.6 * random.random()))
    return price

def generate_material(row):
    """Déduit la matière du produit"""
    materials = {
        'Topwear': ['Coton', 'Polyester', 'Coton bio', 'Mélange coton'],
        'Shoes': ['Cuir', 'Synthétique', 'Toile', 'Mesh'],
        'Watches': ['Acier inoxydable', 'Cuir véritable', 'Silicone'],
        'Bags': ['Cuir', 'Toile', 'Polyester', 'Synthétique'],
        'Bottomwear': ['Jean', 'Coton', 'Polyester', 'Élasthanne'],
        'Jewellery': ['Or plaqué', 'Argent sterling', 'Acier', 'Laiton'],
        'Eyewear': ['Métal', 'Plastique', 'Acétate'],
        'Innerwear': ['Coton', 'Modal', 'Coton élasthanne']
    }
    
    mats = materials.get(row['categorie'], ['Coton', 'Polyester'])
    return random.choice(mats)

print("✅ Fonctions de génération définies (API z.ai via client OpenAI)")

✅ Fonctions de génération définies (API z.ai via client OpenAI)


## 6. Génération par batch avec sauvegarde progressive

In [17]:
# Paramètres
BATCH_SIZE = 20
SAVE_EVERY = 50
START_FROM = 0  # Mettre à X pour reprendre à partir du produit X

# Créer les colonnes si elles n'existent pas
if 'description' not in products_df.columns:
    products_df['description'] = ''
if 'prix' not in products_df.columns:
    products_df['prix'] = 0
if 'matiere' not in products_df.columns:
    products_df['matiere'] = ''

print(f"📊 Génération pour {len(products_df)} produits")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Sauvegarde tous les {SAVE_EVERY} produits")
print(f"   Démarrage à: {START_FROM}")

# Compter les descriptions déjà générées
already_done = products_df['description'].astype(bool).sum()
print(f"   Déjà générées: {already_done}")

# Fonction de sauvegarde
def save_progress(df, filename=None):
    if filename is None:
        filename = PROGRESS_FILE
    output_path = DATA_DIR / filename
    df.to_csv(output_path, index=False)
    print(f"   💾 Sauvegardé: {output_path}")

# Génération
total_to_generate = len(products_df) - START_FROM
failed_generations = []

with tqdm(total=total_to_generate, desc="Génération") as progress_bar:
    for i in range(START_FROM, len(products_df), BATCH_SIZE):
        batch_end = min(i + BATCH_SIZE, len(products_df))
        
        # Traiter le batch
        for idx in range(i, batch_end):
            row = products_df.iloc[idx]
            
            # Skip si déjà généré
            if pd.notna(row['description']) and row['description'] != '':
                progress_bar.update(1)
                continue
            
            # Générer description
            desc = generate_description(row)
            if desc:
                products_df.at[idx, 'description'] = desc
            else:
                failed_generations.append(idx)
            
            # Générer prix et matière (déterministe)
            products_df.at[idx, 'prix'] = generate_price(row)
            products_df.at[idx, 'matiere'] = generate_material(row)
            
            progress_bar.update(1)
        
        # Rate limiting
        time.sleep(1)
        
        # Sauvegarde progressive
        if (batch_end) % SAVE_EVERY == 0:
            save_progress(products_df)

# Sauvegarde finale
save_progress(products_df, OUTPUT_FILE)

print(f"\n✅ Génération terminée !")
print(f"   Descriptions générées: {products_df['description'].astype(bool).sum()}/{len(products_df)}")

if failed_generations:
    print(f"   ⚠️ Échecs: {len(failed_generations)} produits")
    print(f"   Index échoués: {failed_generations[:10]}...")

📊 Génération pour 400 produits
   Batch size: 20
   Sauvegarde tous les 50 produits
   Démarrage à: 0
   Déjà générées: 0


Génération:  25%|██▌       | 100/400 [06:16<20:04,  4.01s/it]

   💾 Sauvegardé: ..\data\products_progress.csv


Génération:  50%|█████     | 200/400 [11:15<21:35,  6.48s/it]

   💾 Sauvegardé: ..\data\products_progress.csv


Génération:  75%|███████▌  | 300/400 [16:22<05:37,  3.38s/it]

   💾 Sauvegardé: ..\data\products_progress.csv


Génération: 100%|██████████| 400/400 [22:02<00:00,  3.31s/it]

   💾 Sauvegardé: ..\data\products_progress.csv
   💾 Sauvegardé: ..\data\products_final.csv

✅ Génération terminée !
   Descriptions générées: 400/400


## 7. Validation et statistiques

In [18]:
# Vérifier les données
print("📊 Validation du dataset final:")
print(f"   Total produits: {len(products_df)}")
print(f"   Descriptions: {products_df['description'].astype(bool).sum()}")
print(f"   Prix: {(products_df['prix'] > 0).sum()}")
print(f"   Matières: {products_df['matiere'].astype(bool).sum()}")

# Aperçu
print("\n🔍 Aperçu des produits générés:")
display_cols = ['nom', 'categorie', 'prix', 'matiere', 'description']
print(products_df[display_cols].head(3).to_string())

# Statistiques des prix
print(f"\n💰 Statistiques des prix:")
print(products_df['prix'].describe())

# Distribution par catégorie
print(f"\n📦 Distribution par catégorie:")
print(products_df['categorie'].value_counts())

📊 Validation du dataset final:
   Total produits: 400
   Descriptions: 400
   Prix: 400
   Matières: 400

🔍 Aperçu des produits générés:
                                     nom categorie   prix        matiere                                                                                                                                                                                                                                                                                          description
0       Visudh Pink Printed Ethnic Kurta   Topwear  19228      Coton bio  Le Visudh Pink Printed Ethnic Kurta est un haut en coton orné d’un imprimé ethnique délicat, pour un look à la fois traditionnel et moderne. Sa coupe ample et son rose vif en font un choix idéal pour une tenue confortable et esthétique, parfaite pour les occasions décontractées ou festives.
1               Femella Women Gold Shirt   Topwear  18138          Coton                                                      La ch

## 8. Vérification des champs vides

In [19]:
# Vérifier les champs vides
missing_desc = products_df[products_df['description'].isna() | (products_df['description'] == '')]
missing_price = products_df[products_df['prix'] == 0]
missing_mat = products_df[products_df['matiere'].isna() | (products_df['matiere'] == '')]

print("⚠️ Champs manquants:")
print(f"   Descriptions vides: {len(missing_desc)}")
print(f"   Prix à 0: {len(missing_price)}")
print(f"   Matières vides: {len(missing_mat)}")

if len(missing_desc) > 0:
    print("\n⚠️ Index avec descriptions manquantes:")
    print(missing_desc.index.tolist())
    print("\n💡 Pour réessayer ces produits, mettez START_FROM =", missing_desc.index[0])

⚠️ Champs manquants:
   Descriptions vides: 0
   Prix à 0: 0
   Matières vides: 0
